# Linear Models for Classification — Exercises

Companion to the note [Linear Models for Classification](Linear%20Models%20for%20Classification.md).

Practise the family of linear classifiers from the note: least squares on 1-of-K targets (and masking), the perceptron, Fisher's discriminant, generative LDA/QDA, one-vs-rest/one-vs-one and linear separability (XOR), by hand and from scratch in NumPy.

**18 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.datasets import load_iris
rng = np.random.default_rng(0)
# Three Gaussian classes on a line (classic masking example)
X3 = np.vstack([rng.normal(size=(100, 2)) + m for m in [(-4, -4), (0, 0), (4, 4)]])
y3 = np.repeat([0, 1, 2], 100)

## Part A · Concepts

### Exercise 1 · Which linear method when?
*🧠 Concept · ★☆☆*

For each scenario, pick one approach from the note's table and justify it in one sentence:
(a) You need class probabilities and want few modelling assumptions.
(b) You have 30 samples per class, 20 features, and the classes look roughly Gaussian with similar spread.
(c) You want to visualise a 10-class problem in 2-D in a supervised way.
(d) Data arrive one at a time on a microcontroller, and they are known to be separable.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Generative models need fewer samples when their assumptions hold; discriminative ones are more robust when they don't.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Logistic regression** (discriminative, models $p(y\mid x)$ directly, see [[Logistic Regression]]).
(b) **LDA**: the shared-covariance Gaussian model has few parameters and is efficient with small samples when the assumption roughly holds.
(c) **Fisher LDA** projects to at most $K-1=9$ dimensions; keep the top 2 discriminant directions.
(d) **Perceptron**: online mistake-driven updates, tiny memory, and it converges because the data are separable.

</details>

### Exercise 2 · Masking in least-squares classification
*🧠 Concept · ★★☆*

Three classes have means on a straight line ($\mu_1, \mu_2, \mu_3$ with $\mu_2$ in the middle). You regress 1-of-K indicator targets
on $x$ with least squares and predict $\arg\max_k \hat y_k(x)$. Explain why the middle class can almost disappear ('masking').

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Project everything on the line through the means. Each $\hat y_k$ is a *linear* function of that coordinate. What does the best line through the middle class's indicator (0, 1, 0 pattern) look like?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Along the line, the indicator of class 2 is $0$ at the left, $1$ in the middle, $0$ at the right, a bump. The best *linear* fit to a bump is nearly flat
(about $1/3$), while class 1's fitted score decreases and class 3's increases steeply. So $\hat y_2$ is almost never the largest, and the middle
class is masked. Fixes: quadratic features, or a probabilistic model (LDA / logistic regression), which do not have this problem. You will see it in Exercise 13.

</details>

### Exercise 3 · XOR is not linearly separable
*🧠 Concept · ★☆☆*

Points $(0,0),(1,1)$ have label $-1$ and $(0,1),(1,0)$ label $+1$. Prove that no $(w_1,w_2,w_0)$ classifies all four correctly with $\operatorname{sign}(w^\top x+w_0)$,
and give a single extra feature $\phi_3(x)$ that makes the data linearly separable.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Write the four inequalities and add the two for the $+1$ class, then the two for the $-1$ class.

</details>

<details>
<summary><b>✅ Solution</b></summary>

We need $w_0<0$, $w_1+w_2+w_0<0$ (class $-1$) and $w_2+w_0>0$, $w_1+w_0>0$ (class $+1$).
Adding the last two: $w_1+w_2+2w_0>0$. Adding the first two: $w_1+w_2+2w_0<0$. Contradiction.
With $\phi_3=x_1x_2$ (or $(x_1-x_2)^2$), e.g. $f=x_1+x_2-2x_1x_2-\tfrac12$ gives $-\tfrac12, +\tfrac12, +\tfrac12, -\tfrac12$. ✔
This is the motivation for basis functions, [[Kernel Methods]] and [[Neural Networks]].

</details>

### Exercise 4 · LDA, QDA or logistic regression?
*🧠 Concept · ★★☆*

(a) Why does a *shared* covariance give a linear boundary, while separate covariances give a quadratic one?
(b) Both LDA and logistic regression produce $p(y=1\mid x)=\sigma(w^\top x+w_0)$. Why can they still give different $w$?
(c) Count parameters for QDA vs LDA with $K$ classes and $d$ features.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

(c) Each covariance matrix has $d(d+1)/2$ free entries.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\log p(x\mid k)$ contains $-\tfrac12x^\top\Sigma_k^{-1}x$. With a shared $\Sigma$ this term is the same for all classes and cancels in $\delta_k-\delta_j$; with $\Sigma_k$ it does not, leaving a quadratic in $x$.
(b) LDA fits means/covariance by maximising the **joint** likelihood $p(x,y)$; logistic regression maximises the **conditional** likelihood $p(y\mid x)$.
When the Gaussian assumption is wrong (outliers, non-Gaussian features), the estimates differ, and logistic regression is usually more robust.
(c) LDA: $Kd$ means $+\,d(d+1)/2$ covariance $+\,K-1$ priors. QDA: $Kd+K\,d(d+1)/2+K-1$. For $d=50,K=3$: 1427 vs 3977.

</details>

## Part B · By hand

### Exercise 5 · Counting classifiers: OvR vs OvO
*✍️ By hand · ★☆☆*

How many binary classifiers do one-vs-rest and one-vs-one need for $K=5$ classes? For $K=10$? Store the four numbers.
Why can OvO still be cheaper to train than OvR?

In [ ]:
ovr5 = None
ovo5 = None
ovr10 = None
ovo10 = None

from math import comb
check('OvR K=5', ovr5, 5)
check('OvO K=5', ovo5, comb(5, 2))
check('OvR K=10', ovr10, 10)
check('OvO K=10', ovo10, comb(10, 2))

<details>
<summary><b>💡 Hint</b></summary>

OvO trains one classifier per unordered pair of classes.

</details>

<details>
<summary><b>✅ Solution</b></summary>

OvR: $K$ classifiers (5 and 10). OvO: $\binom K2=K(K-1)/2$ (10 and 45). Each OvO classifier only sees the data of 2 classes (about $2N/K$ points),
so for learners with super-linear cost (e.g. kernel SVMs, $O(N^2)$–$O(N^3)$) the total OvO cost can be lower.

```python
ovr5, ovo5, ovr10, ovo10 = 5, 10, 10, 45
```

</details>

### Exercise 6 · Perceptron by hand
*✍️ By hand · ★☆☆*

Run the perceptron $w\leftarrow w+y_ix_i$ (update when $y_iw^\top x_i\le0$) starting from $w=0$, cycling through the points in order.
Each $x$ already includes a leading 1 for the bias.

| $x$ | $(1,2,1)$ | $(1,0,-1)$ | $(1,-1,1)$ | $(1,-2,-1)$ |
|---|---|---|---|---|
| $y$ | $+1$ | $+1$ | $-1$ | $-1$ |

Store the final weight vector and the total number of updates.

In [ ]:
w_perc = None
n_updates = None

_X = np.array([[1,2,1],[1,0,-1],[1,-1,1],[1,-2,-1.]]); _y = np.array([1,1,-1,-1])
_w = np.zeros(3); _n = 0
for _ in range(20):
    for _xi, _yi in zip(_X, _y):
        if _yi * (_w @ _xi) <= 0:
            _w = _w + _yi * _xi; _n += 1
check('final w', w_perc, _w)
check('number of updates', n_updates, _n)

<details>
<summary><b>💡 Hint</b></summary>

Note the rule is '$\le 0$', so a point exactly on the boundary ($w^\top x=0$) also triggers an update.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Epoch 1: $x_1$: $w^\top x=0$ → $w=(1,2,1)$. $x_2$: $1+0-1=0$ → $w=(2,2,0)$. $x_3$: $2-2+0=0$ → $w=(2,2,0)-(1,-1,1)=(1,3,-1)$.
$x_4$: $1-6+1=-4$, $y=-1$ → correct.
Epoch 2: margins $y\,w^\top x = 6, 2, 3, 4$, all positive: converged. Final $w=(1,3,-1)$ after **3** updates.

```python
w_perc = [1, 3, -1]
n_updates = 3
```

</details>

### Exercise 7 · Perceptron convergence bound
*✍️ By hand · ★★☆*

Novikoff's theorem: if $\lVert x_i\rVert\le R$ and some unit vector $u$ achieves $y_iu^\top x_i\ge\gamma>0$ for all $i$, the perceptron makes at most
$(R/\gamma)^2$ mistakes. (a) Sketch the proof using $u^\top w_t$ and $\lVert w_t\rVert^2$. (b) Evaluate the bound for $R=3$, $\gamma=0.5$.

In [ ]:
mistake_bound = None

check('bound', mistake_bound, (3/0.5)**2)

<details>
<summary><b>💡 Hint 1</b></summary>

Each mistake increases $u^\top w$ by at least $\gamma$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Each mistake increases $\lVert w\rVert^2$ by at most $R^2$, because $2y_iw^\top x_i\le0$ on a mistake. Combine with Cauchy–Schwarz.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) After $t$ mistakes: $u^\top w_t=u^\top w_{t-1}+y_iu^\top x_i\ge t\gamma$, and
$\lVert w_t\rVert^2=\lVert w_{t-1}\rVert^2+2y_iw_{t-1}^\top x_i+\lVert x_i\rVert^2\le tR^2$.
Cauchy–Schwarz: $t\gamma\le u^\top w_t\le\lVert w_t\rVert\le\sqrt t R$, so $t\le (R/\gamma)^2$.
(b) $(3/0.5)^2=36$ mistakes at most, independent of $N$ and of the dimension. If the data are not separable there is no $\gamma>0$ and the perceptron never stops.

```python
mistake_bound = 36
```

</details>

### Exercise 8 · Fisher's criterion
*✍️ By hand · ★★☆*

Fisher chooses $w$ to maximise $J(w)=\dfrac{w^\top S_Bw}{w^\top S_Ww}$ with $S_B=(m_2-m_1)(m_2-m_1)^\top$ and
$S_W=\sum_{n\in C_1}(x_n-m_1)(x_n-m_1)^\top+\sum_{n\in C_2}(x_n-m_2)(x_n-m_2)^\top$. Show that $w\propto S_W^{-1}(m_2-m_1)$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Set $\nabla J=0$ with the quotient rule. Note $S_Bw$ always points along $m_2-m_1$, and the length of $w$ does not matter.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla J=0 \iff (w^\top S_Bw)\,S_Ww=(w^\top S_Ww)\,S_Bw$. Since $S_Bw=(m_2-m_1)\big[(m_2-m_1)^\top w\big]$ is a scalar multiple of $m_2-m_1$,
we get $S_Ww\propto m_2-m_1$, i.e. $w\propto S_W^{-1}(m_2-m_1)$. Only the direction matters ($J$ is scale invariant).
If $S_W\propto I$ this is simply the difference of the means; otherwise $S_W^{-1}$ rotates $w$ away from directions where classes are spread out.

</details>

### Exercise 9 · Fisher's discriminant with numbers
*✍️ By hand · ★★☆*

Class 1: $(1,2),(2,3),(3,3)$. Class 2: $(5,5),(6,7),(7,6)$. Compute $m_1,m_2$, $S_W$ and $w=S_W^{-1}(m_2-m_1)$.

In [ ]:
m1_f = None
m2_f = None
SW_f = None
w_f = None

_A = np.array([[1,2],[2,3],[3,3.]]); _B = np.array([[5,5],[6,7],[7,6.]])
_SW = (_A-_A.mean(0)).T@(_A-_A.mean(0)) + (_B-_B.mean(0)).T@(_B-_B.mean(0))
check('m1', m1_f, _A.mean(0)); check('m2', m2_f, _B.mean(0)); check('S_W', SW_f, _SW)
check('w', w_f, np.linalg.solve(_SW, _B.mean(0)-_A.mean(0)))

<details>
<summary><b>💡 Hint 1</b></summary>

$m_1=(2, 8/3)$, $m_2=(6,6)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Class 1 deviations: $(-1,-2/3),(0,1/3),(1,1/3)$; class 2: $(-1,-1),(0,1),(1,0)$. Sum the outer products.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Class 1 scatter: $\begin{pmatrix}2&1\\1&2/3\end{pmatrix}$, class 2 scatter: $\begin{pmatrix}2&1\\1&2\end{pmatrix}$,
so $S_W=\begin{pmatrix}4&2\\2&8/3\end{pmatrix}$, $\det S_W=20/3$.
$m_2-m_1=(4,10/3)$, $S_W^{-1}=\frac{3}{20}\begin{pmatrix}8/3&-2\\-2&4\end{pmatrix}$, so
$w=\frac3{20}\,(32/3-20/3,\ -8+40/3)=\frac3{20}(4, 16/3)=(0.6,\ 0.8)$.

```python
m1_f = [2, 8/3]
m2_f = [6, 6]
SW_f = [[4, 2], [2, 8/3]]
w_f = [0.6, 0.8]
```

</details>

### Exercise 10 · Evaluating LDA discriminants
*✍️ By hand · ★☆☆*

Two classes share $\Sigma=\operatorname{diag}(2,1)$, with $\mu_1=(0,0)$, $\mu_2=(2,2)$, $\pi_1=0.7$, $\pi_2=0.3$.
Using $\delta_k(x)=x^\top\Sigma^{-1}\mu_k-\tfrac12\mu_k^\top\Sigma^{-1}\mu_k+\log\pi_k$, compute $\delta_1,\delta_2$ at $x=(1,1.2)$ and the predicted class.

In [ ]:
delta1 = None
delta2 = None
k_pred = None  # 1 or 2

_Si = np.linalg.inv(np.diag([2., 1.])); _x = np.array([1, 1.2])
_d = [_x@_Si@m - 0.5*m@_Si@m + np.log(p) for m, p in [(np.zeros(2), .7), (np.array([2, 2.]), .3)]]
check('delta1', delta1, _d[0], tol=1e-3); check('delta2', delta2, _d[1], tol=1e-3)
check('class', k_pred, int(np.argmax(_d)) + 1)

<details>
<summary><b>💡 Hint</b></summary>

$\Sigma^{-1}\mu_2=(1,2)$ and $\mu_2^\top\Sigma^{-1}\mu_2=6$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\delta_1=0-0+\log0.7\approx-0.3567$. $\delta_2=(1)(1)+(1.2)(2)-3+\log0.3=3.4-3-1.2040\approx-0.8040$.
$\delta_1>\delta_2$, so class 1. (With equal priors $\delta_2=0.4-0.693=-0.293>\delta_1=-0.693$ and the prediction would flip: priors matter near the boundary.)

```python
delta1 = np.log(0.7)
delta2 = 3.4 - 3 + np.log(0.3)
k_pred = 1
```

</details>

### Exercise 11 · Deriving the LDA discriminant
*✍️ By hand · ★★☆*

Starting from $p(x\mid k)=\mathcal N(x\mid\mu_k,\Sigma)$ and prior $\pi_k$, show that $\arg\max_k p(k\mid x)=\arg\max_k\delta_k(x)$ with the
$\delta_k$ of the note. Then show that for two classes the posterior is $\sigma(w^\top x+w_0)$ with $w=\Sigma^{-1}(\mu_1-\mu_2)$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Take $\log\big(p(x\mid k)\pi_k\big)$, expand the quadratic form, and drop every term that does not depend on $k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log p(x\mid k)\pi_k=-\tfrac12(x-\mu_k)^\top\Sigma^{-1}(x-\mu_k)+\log\pi_k+c
=-\tfrac12x^\top\Sigma^{-1}x+x^\top\Sigma^{-1}\mu_k-\tfrac12\mu_k^\top\Sigma^{-1}\mu_k+\log\pi_k+c$.
The first term and $c$ are the same for every $k$, so maximising the posterior equals maximising $\delta_k$.
Two classes: $p(1\mid x)=\frac{1}{1+e^{-(\delta_1-\delta_2)}}=\sigma(a)$ with
$a=\delta_1-\delta_2=x^\top\Sigma^{-1}(\mu_1-\mu_2)-\tfrac12\mu_1^\top\Sigma^{-1}\mu_1+\tfrac12\mu_2^\top\Sigma^{-1}\mu_2+\log\frac{\pi_1}{\pi_2}$,
which is linear in $x$: the same functional form as [[Logistic Regression]].

</details>

## Part C · Code from scratch

### Exercise 12 · The perceptron
*💻 Code · ★☆☆*

Implement `perceptron(X, y, max_epochs)` for labels $y\in\{-1,+1\}$ (add the bias column inside). Return the weights and the number of
epochs until an epoch with no mistakes. Run it on the separable data below.

In [ ]:
def perceptron(X, y, max_epochs=100):
    # TODO: return (w, epochs)  with w = [w0, w1, ...]
    return None, None

Xp = rng.normal(size=(100, 2))
yp = np.where(Xp @ np.array([1.0, -2.0]) + 0.3 > 0, 1, -1)
Xp = Xp + 0.3 * yp[:, None] * np.array([1.0, -2.0]) / np.sqrt(5)   # push classes apart: margin >= 0.3
w_p, ep_p = perceptron(Xp, yp)

check('all training points correct', None if w_p is None else np.all(np.sign(np.c_[np.ones(len(Xp)), Xp] @ w_p) == yp), True)

<details>
<summary><b>💡 Hint</b></summary>

Loop over epochs; inside, loop over points and update on `y_i * w @ x_i <= 0`. Stop when an epoch makes no updates.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The perceptron finds *a* separating hyperplane, not the best one: the result depends on data order and initialisation.
The max-margin choice is what [[Support Vector Machines]] add.

```python
def perceptron(X, y, max_epochs=100):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for ep in range(1, max_epochs + 1):
        mistakes = 0
        for xi, yi in zip(Xb, y):
            if yi * (w @ xi) <= 0:
                w += yi * xi
                mistakes += 1
        if mistakes == 0:
            return w, ep
    return w, max_epochs

Xp = rng.normal(size=(100, 2))
yp = np.where(Xp @ np.array([1.0, -2.0]) + 0.3 > 0, 1, -1)
Xp = Xp + 0.3 * yp[:, None] * np.array([1.0, -2.0]) / np.sqrt(5)
w_p, ep_p = perceptron(Xp, yp)
print('weights', w_p, 'epochs', ep_p)
```

</details>

### Exercise 13 · Least squares on 1-of-K targets: masking
*💻 Code · ★★☆*

Fit least squares to one-hot targets for the shared three-class data `X3, y3` (with bias) and predict with $\arg\max$.
Store the overall training accuracy in `acc_ls` and the accuracy *on the middle class only* in `acc_mid`. Compare with sklearn's LDA.

In [ ]:
acc_ls = None
acc_mid = None

_lda = LinearDiscriminantAnalysis().fit(X3, y3)
print('LDA accuracy:', _lda.score(X3, y3))
_Xb = np.c_[np.ones(300), X3]; _p = (_Xb @ np.linalg.lstsq(_Xb, np.eye(3)[y3], rcond=None)[0]).argmax(1)
check('LS accuracy', acc_ls, np.mean(_p == y3))
check('middle-class accuracy', acc_mid, np.mean(_p[y3 == 1] == 1))

<details>
<summary><b>💡 Hint</b></summary>

`W = np.linalg.lstsq(Xb, np.eye(3)[y3], rcond=None)[0]`, then `(Xb @ W).argmax(1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Least squares gets only about 70% overall and under 10% on the middle class: it is masked (Exercise 2). LDA, with the same linear boundaries
but fitted probabilistically, gets about 99.7%.

```python
Xb3 = np.c_[np.ones(len(X3)), X3]
W_ls = np.linalg.lstsq(Xb3, np.eye(3)[y3], rcond=None)[0]
pred_ls = (Xb3 @ W_ls).argmax(1)
acc_ls = np.mean(pred_ls == y3)
acc_mid = np.mean(pred_ls[y3 == 1] == 1)
print(acc_ls, acc_mid)
```

</details>

### Exercise 14 · Fisher's discriminant direction
*💻 Code · ★★☆*

Implement `fisher_direction(X, y)` for two classes (labels 0/1) returning the unit vector $w\propto S_W^{-1}(m_1-m_0)$.
Apply it to Iris classes 1 and 2 (versicolor vs virginica, all 4 features) and compare with the direction of sklearn's `LinearDiscriminantAnalysis().coef_`.

In [ ]:
def fisher_direction(X, y):
    # TODO
    return None

Xir, yir = load_iris(return_X_y=True)
mask = yir > 0
X_vv, y_vv = Xir[mask], (yir[mask] == 2).astype(int)
w_fisher = fisher_direction(X_vv, y_vv)

_c = LinearDiscriminantAnalysis().fit(X_vv, y_vv).coef_.ravel(); _c = _c / np.linalg.norm(_c)
check('|cos(w, sklearn)| = 1', None if w_fisher is None else abs(np.dot(w_fisher, _c)) / np.linalg.norm(w_fisher), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

`S_W = (X0-m0).T @ (X0-m0) + (X1-m1).T @ (X1-m1)`; solve instead of inverting.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The directions coincide: for two classes, Fisher's discriminant and generative LDA produce the same projection direction
(the pooled covariance is $S_W/(N-2)$, a scalar multiple). They only differ in how the threshold is chosen.

```python
def fisher_direction(X, y):
    X0, X1 = X[y == 0], X[y == 1]
    m0, m1 = X0.mean(0), X1.mean(0)
    SW = (X0 - m0).T @ (X0 - m0) + (X1 - m1).T @ (X1 - m1)
    w = np.linalg.solve(SW, m1 - m0)
    return w / np.linalg.norm(w)

Xir, yir = load_iris(return_X_y=True)
mask = yir > 0
X_vv, y_vv = Xir[mask], (yir[mask] == 2).astype(int)
w_fisher = fisher_direction(X_vv, y_vv)
print(w_fisher)
```

</details>

### Exercise 15 · Generative LDA classifier
*💻 Code · ★★☆*

Implement `lda_fit(X, y)` (class means, pooled covariance with denominator $N-K$, priors) and `lda_predict(params, X)` using the
note's $\delta_k(x)$. Check that your predictions on full Iris match sklearn's `LinearDiscriminantAnalysis` exactly.

In [ ]:
def lda_fit(X, y):
    # TODO: return (means [K,d], Sigma [d,d], priors [K])
    return None

def lda_predict(params, X):
    # TODO: argmax_k delta_k(x)
    return None

lda_params = lda_fit(Xir, yir)
pred_lda = None if lda_params is None else lda_predict(lda_params, Xir)

check('agreement with sklearn', None if pred_lda is None else np.mean(pred_lda == LinearDiscriminantAnalysis().fit(Xir, yir).predict(Xir)), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

With `Si = inv(Sigma)`: `D = X @ Si @ mu.T - 0.5 * np.sum(mu @ Si * mu, 1) + np.log(pi)` gives all $\delta_k$ at once.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Training is just counting means and one covariance (closed form, no iterations). The vectorised discriminant evaluates all classes at once.
The pooled-covariance denominator does not change predictions much, but $N-K$ matches sklearn.

```python
def lda_fit(X, y):
    classes = np.unique(y)
    mu = np.array([X[y == k].mean(0) for k in classes])
    S = sum((X[y == k] - mu[i]).T @ (X[y == k] - mu[i]) for i, k in enumerate(classes))
    Sigma = S / (len(X) - len(classes))
    pi = np.array([np.mean(y == k) for k in classes])
    return mu, Sigma, pi

def lda_predict(params, X):
    mu, Sigma, pi = params
    Si = np.linalg.inv(Sigma)
    D = X @ Si @ mu.T - 0.5 * np.sum(mu @ Si * mu, axis=1) + np.log(pi)
    return D.argmax(1)

lda_params = lda_fit(Xir, yir)
pred_lda = lda_predict(lda_params, Xir)
```

</details>

### Exercise 16 · QDA from scratch
*💻 Code · ★★★*

Implement QDA: per class $\mu_k$, $\Sigma_k$ (denominator $n_k-1$) and $\pi_k$, with discriminant
$\delta_k(x)=-\tfrac12\log|\Sigma_k|-\tfrac12(x-\mu_k)^\top\Sigma_k^{-1}(x-\mu_k)+\log\pi_k$.
On the data below (one tight class inside a wide one), compare training accuracy with LDA and check that your predictions match sklearn's QDA.

In [ ]:
def qda_fit(X, y):
    # TODO: return list of (mu_k, Sigma_k, pi_k)
    return None

def qda_predict(params, X):
    # TODO
    return None

Xq = np.vstack([rng.normal(scale=0.5, size=(150, 2)), rng.normal(scale=2.5, size=(150, 2))])
yq = np.repeat([0, 1], 150)
qda_params = qda_fit(Xq, yq)
pred_qda = None if qda_params is None else qda_predict(qda_params, Xq)

print('LDA accuracy:', LinearDiscriminantAnalysis().fit(Xq, yq).score(Xq, yq))
check('agreement with sklearn QDA', None if pred_qda is None else np.mean(pred_qda == QuadraticDiscriminantAnalysis().fit(Xq, yq).predict(Xq)), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.slogdet(S)[1]` gives $\log|\Sigma_k|$ stably; `np.einsum('ij,jk,ik->i', D, Si, D)` evaluates the quadratic forms for all rows.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The classes have the same mean, so every linear boundary is nearly useless (LDA ≈ 62%), while QDA's circular boundary separates the
tight core from the wide ring (≈ 93%). This is the 'separate covariances ⇒ quadratic' row of the note in action.

```python
def qda_fit(X, y):
    out = []
    for k in np.unique(y):
        Xk = X[y == k]
        out.append((Xk.mean(0), np.cov(Xk.T, ddof=1), len(Xk) / len(X)))
    return out

def qda_predict(params, X):
    scores = []
    for mu, S, pi in params:
        D = X - mu
        quad = np.einsum('ij,jk,ik->i', D, np.linalg.inv(S), D)
        scores.append(-0.5 * np.linalg.slogdet(S)[1] - 0.5 * quad + np.log(pi))
    return np.argmax(np.array(scores), axis=0)

Xq = np.vstack([rng.normal(scale=0.5, size=(150, 2)), rng.normal(scale=2.5, size=(150, 2))])
yq = np.repeat([0, 1], 150)
qda_params = qda_fit(Xq, yq)
pred_qda = qda_predict(qda_params, Xq)
print('QDA accuracy:', np.mean(pred_qda == yq))
```

</details>

### Exercise 17 · Solving XOR with a feature map
*💻 Code · ★☆☆*

Run your `perceptron` on the four XOR points (labels $\pm1$) with raw features, then with $\phi(x)=(x_1,x_2,x_1x_2)$.
Store the training accuracy of both in `acc_raw` and `acc_phi`.

In [ ]:
X_xor = np.array([[0, 0], [1, 1], [0, 1], [1, 0.]])
y_xor = np.array([-1, -1, 1, 1])
acc_raw = None
acc_phi = None

check('raw features cannot reach 100%', None if acc_raw is None else acc_raw <= 0.75, True)
check('with x1*x2 the perceptron separates XOR', acc_phi, 1.0)

<details>
<summary><b>💡 Hint</b></summary>

Use `max_epochs=1000` for the raw features: the perceptron never converges, so just evaluate the final weights.

</details>

<details>
<summary><b>✅ Solution</b></summary>

On raw features the perceptron cycles forever and its final weights get at most 3/4 right (Exercise 3 shows 4/4 is impossible).
With the product feature it converges in about ten epochs to 100%. A kernel computes such products implicitly ([[Kernel Methods]]).

```python
X_xor = np.array([[0, 0], [1, 1], [0, 1], [1, 0.]])
y_xor = np.array([-1, -1, 1, 1])
def _acc(w, X, y):
    return np.mean(np.sign(np.c_[np.ones(len(X)), X] @ w) == y)
w_raw, _ = perceptron(X_xor, y_xor, max_epochs=1000)
Phi_xor = np.c_[X_xor, X_xor[:, 0] * X_xor[:, 1]]
w_phi, ep = perceptron(Phi_xor, y_xor)
acc_raw, acc_phi = _acc(w_raw, X_xor, y_xor), _acc(w_phi, Phi_xor, y_xor)
print(acc_raw, acc_phi, 'epochs with phi:', ep)
```

</details>

### Exercise 18 · One-vs-rest by hand-rolled code
*💻 Code · ★★☆*

Implement `ovr_fit(X, y)` that trains one `LogisticRegression()` per class ($k$ vs rest) and `ovr_predict(models, X)` that predicts the class whose
model gives the highest `decision_function`. Check that on Iris your predictions agree with `OneVsRestClassifier(LogisticRegression())`.
Why do we use the continuous scores rather than hard 0/1 votes?

In [ ]:
def ovr_fit(X, y):
    # TODO: list of fitted binary models
    return None

def ovr_predict(models, X):
    # TODO
    return None

ovr_models = ovr_fit(Xir, yir)
pred_ovr = None if ovr_models is None else ovr_predict(ovr_models, Xir)

_ref = OneVsRestClassifier(LogisticRegression()).fit(Xir, yir).predict(Xir)
check('agreement with sklearn OvR', None if pred_ovr is None else np.mean(pred_ovr == _ref), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

`np.column_stack([m.decision_function(X) for m in models]).argmax(1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With hard votes, a point may be claimed by several binary classifiers or by none; these are the *ambiguous regions* of the note's table.
Taking $\arg\max$ of the real-valued scores removes them (at the cost of assuming the scores are comparable across classifiers).

```python
def ovr_fit(X, y):
    return [LogisticRegression().fit(X, (y == k).astype(int)) for k in np.unique(y)]

def ovr_predict(models, X):
    return np.column_stack([m.decision_function(X) for m in models]).argmax(1)

ovr_models = ovr_fit(Xir, yir)
pred_ovr = ovr_predict(ovr_models, Xir)
print('accuracy', np.mean(pred_ovr == yir))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Linear Models for Classification](Linear%20Models%20for%20Classification.md).*